# Repeated requests at one charging point

- **Method:** [protocol.ipynb](protocol.ipynb), Sections II–III, loaded unchanged as in
  [charging_morning.ipynb](charging_morning.ipynb). No method code is copied here.
- **Principal test:** commitments remain valid across successive negotiations. After **every** attempt,
  direct interval-overlap arithmetic and complete-trace evaluation check the method's output,
  independently of the contract simplifier.
- **Workloads are declared synthetic.** Agreement rates illustrate behaviour under these workloads and
  policies only. Failures are classified so that sampling or bargaining failure stays distinct from
  infeasible service.
- **Run:** `uv run --group notebooks python scripts/run_protocol_notebook.py --notebook repeated_requests --save`
  (a few minutes; timings depend on the machine).

In [1]:
from collections import Counter
from fractions import Fraction as F
from pathlib import Path
from statistics import median, quantiles
from time import perf_counter
import random
import portion as P
import interval_logic as il
from protocol import (Booking, Request, Service, standing, affine_policies, decide_booking,
                      independent_feasible_starts, joint_witness, table, display_timeline)
from protocol.companion import load_method_notebook, clock, clock_interval, plot_outcome_shares

NOTEBOOK_DIR = Path('notebooks') if Path('notebooks').is_dir() else Path('.')
method = load_method_notebook(NOTEBOOK_DIR / 'protocol.ipynb')

## Declared workload

The clock is in minutes since midnight. Start windows are **closed**; bookings are half-open.

| Parameter | Values |
|---|---|
| Booking horizon | 08:00–18:00, initially A 08:00–09:00 and B 10:30–11:30 (the illustrated example) |
| Requests per day (load) | 10, 20, 40 |
| Requested duration $d$ | 30, 60, 90 min, uniform |
| Start flexibility $w$ | 30, 60, 120 min, uniform; window $J=[a,a+w]$ |
| Earliest start $a$ | 5-min grid, uniform over $a\geq$ 08:00 and $a+w+d\leq$ 18:00 |
| Receipt | 5–180 min before $a$ (5-min grid), so every requested start is in the future |
| Permitted adaptation | exactly half the requests, drawn beforehand: one alternative, same $J$, duration $d/2$ |
| Repetitions | 20 seeds per load level; each seed fixes the whole day |

Each request comes from a new client (ids 3, 4, … by receipt). Requests are processed in order of receipt,
ties in draw order. Each attempt starts from the commitments left by the previous attempt.

**Bargaining, fixed throughout:** $u_o=q-c$, $u_c=v-q$; earliest legal offer meeting the proposer's target;
target-based acceptance; halfway concession; owner opens; three rounds; prices €0–€12;
targets $(4,4)$, minima $(1,1)$. Per request, $c\in\{1,1.5,\dots,4\}$ and $v\in\{4,4.5,\dots,10\}$, uniform, the same for
every offer. An adapted service is valued like the original (declared simplification).

- **Attainability:** $c+4\leq12$ and $v-4\geq0$, so both initial targets are attainable individually.
- **Acceptable ranges:** $[c+1,12]$ and $[0,v-1]$ overlap iff $v-c\geq2$.
- **Agreement:** these policies need $v-c\geq7/2$ (acceptance by round 3 at the latest).
  Overlapping ranges with $2\leq v-c<7/2$ exhaust the budget.

In [2]:
NAMES = {1: 'A', 2: 'B'}
OPEN_TIME, CLOSE_TIME = 480, 1080
INITIAL = (Booking(1, P.closedopen(480, 540)), Booking(2, P.closedopen(630, 690)))
LOADS, SEEDS = (10, 20, 40), range(20)
DURATIONS, FLEXIBILITY, LEAD_TIMES = (30, 60, 90), (30, 60, 120), range(5, 181, 5)
PRICES, ROUND_BUDGET = (F(0), F(12)), 3
TARGETS = {'owner': F(4), 'client': F(4)}
MINIMA = {'owner': F(1), 'client': F(1)}
COSTS = tuple(F(k, 2) for k in range(2, 9))    # €1–€4
VALUES = tuple(F(k, 2) for k in range(8, 21))  # €4–€10
OUTCOMES = ('Agreement · full service', 'Agreement · adapted service', 'Failure · infeasible service',
            'Failure · empty sampled offer set', 'Failure · exhausted bargaining')

In [3]:
def synthetic_day(load, seed):
    # Draw every request first; then order by receipt and number clients accordingly.
    rng = random.Random(1000 * load + seed)
    adapted = set(rng.sample(range(load), load // 2))
    draws = []
    for i in range(load):
        d, w = rng.choice(DURATIONS), rng.choice(FLEXIBILITY)
        a = rng.randrange(OPEN_TIME, CLOSE_TIME - d - w + 1, 5)
        J = P.closed(a, a + w)
        draws.append(dict(received=F(a - rng.choice(LEAD_TIMES)), order=i, service=Service(J, d),
                          alternatives=(Service(J, F(d, 2)),) if i in adapted else (),
                          cost=rng.choice(COSTS), value=rng.choice(VALUES)))
    draws.sort(key=lambda r: (r['received'], r['order']))
    specs = tuple(dict(received=r['received'], request=Request(3 + k, r['service'], r['alternatives']),
                       cost=r['cost'], value=r['value'], targets=TARGETS, minima=MINIMA)
                  for k, r in enumerate(draws))

    # Declared workload properties, checked on the generated data.
    for s in specs:
        services = (s['request'].service, *s['request'].alternatives)
        assert all(OPEN_TIME <= v.starts.lower and s['received'] < v.starts.lower
                   and v.starts.upper + v.duration <= CLOSE_TIME for v in services)
        assert s['cost'] + TARGETS['owner'] <= PRICES[1] and s['value'] - TARGETS['client'] >= PRICES[0]
    assert sum(bool(s['request'].alternatives) for s in specs) == load // 2
    return specs

## One attempt and its independent checks

`attempt` runs the unchanged `admit → generate → bargain → commit` methods and times only those four calls.
The checks then use **direct calculations**. Half-open $[a,b)$ and $[c,e)$ overlap iff $a<e$ and $c<b$;
`independent_feasible_starts` tests critical-boundary partitions this way, without pruning or normalization.
Complete-trace evaluation (`il.satisfies`) is used, not the contract simplifier.

- **Feasibility agreement:** admission selects the first service, in order, with independently feasible
  starts. It returns exactly those starts, or `None` when no service has any. Every offer starts there and
  overlaps no other client's booking.
- **Retained:** earlier bookings and ledger form an unchanged prefix. The new commitment is exactly the
  old commitment conjoined with the accepted booking, i.e. the conjunction of all retained bookings.
- **Jointly satisfiable:** no two clients' bookings overlap. The access trace built from them satisfies the
  commitment, and the production decision agrees.
- **Selected service:** each agreement is for the requesting client, with the selected service's duration,
  a start in its window, not in the past. On the trace it witnesses that service's $S_a$.
- **Minimum utilities:** the accepted price $q\in[0,12]$ has $q-c\geq1$ and $v-q\geq1$, using the declared
  $c,v$, not the policy callbacks.
- **Failure unchanged:** a failed attempt leaves commitment, ledger and bookings identical.
- **Budget:** at most three rounds. An agreement is accepted only in its final round; exhausted
  bargaining uses all three rounds; no offers means no rounds.

In [4]:
def overlaps(x, y):
    return x.lower < y.upper and y.lower < x.upper


def independent_selection(request, bookings, now):
    for service in (request.service, *request.alternatives):
        starts = independent_feasible_starts(service, bookings, request.client, now)
        if not starts.empty:
            return service, starts
    return None, P.empty()


def classify(request, admission, offers, negotiation):
    if admission is None:
        return OUTCOMES[2]
    if not offers:
        return OUTCOMES[3]
    if negotiation.agreement is None:
        return OUTCOMES[4]
    return OUTCOMES[0] if admission.service == request.service else OUTCOMES[1]


def check_attempt(spec, admission, offers, negotiation, result, commitment, ledger, before, after):
    request, now, agreement = spec['request'], spec['received'], negotiation.agreement

    # Admission and compatible starts, by direct interval-overlap calculation.
    service, starts = independent_selection(request, before, now)
    disagreement = (((admission is None) != (service is None))
                    or (admission is not None and (admission.service, admission.feasible_starts) != (service, starts))
                    or any(o.start not in starts or any(overlaps(o.interval, b.interval)
                                                        for b in before if b.client != request.client)
                           for o in offers))

    # Joint satisfiability: pairwise disjoint across clients, a satisfying trace, and the decision.
    disjoint = not any(overlaps(x.interval, y.interval) for i, x in enumerate(after)
                       for y in after[i + 1:] if x.client != y.client)
    trace = joint_witness(after, ()) if disjoint else None
    satisfiable = (disjoint and il.satisfies(result.commitment.guarantee, trace, 0)
                   and decide_booking(result.commitment, now).satisfiable)

    if agreement is None:
        provides = minimum = True
    else:
        b, q = agreement.offer.booking, agreement.price
        selected = admission.service
        provides = (selected in (request.service, *request.alternatives) and b.client == request.client
                    and b.duration == selected.duration and b.start in selected.starts and b.start >= now
                    and trace is not None
                    and il.satisfies(method.service_contracts(request.client, selected)[0], trace, now))
        minimum = (PRICES[0] <= q <= PRICES[1] and q - spec['cost'] >= spec['minima']['owner']
                   and spec['value'] - q >= spec['minima']['client'])

    accepted = [r.accepted for r in negotiation.history]
    in_budget = len(accepted) <= ROUND_BUDGET and (
        (not offers and not accepted)
        or (agreement is not None and accepted == [False] * (len(accepted) - 1) + [True])
        or (agreement is None and len(accepted) == ROUND_BUDGET and not any(accepted)))

    expected = commitment if agreement is None else commitment.conjoin(agreement.offer.contract)
    violations = {
        'retained': (after[:len(before)] != before or result.ledger[:len(ledger)] != ledger
                     or result.commitment != expected or result.commitment != standing(*after)),
        'jointly satisfiable': not satisfiable,
        'selected service': not provides,
        'minimum utilities': not minimum,
        'failure unchanged': agreement is None and (result.commitment != commitment
                                                    or result.ledger != ledger or after != before),
        'within budget': not in_budget,
    }
    return dict(disagreement=disagreement, violations=violations)


def attempt(spec, commitment, bookings, ledger):
    request, now = spec['request'], spec['received']
    started = perf_counter()
    admission = method.admit(request, commitment, now)
    offers = method.generate(admission)
    policies = affine_policies({o: spec['cost'] for o in offers}, {o: spec['value'] for o in offers})
    negotiation = method.bargain(offers, policies, spec['targets'], spec['minima'], PRICES, ROUND_BUDGET)
    result = method.commit(admission, offers, negotiation, commitment, ledger)
    seconds = perf_counter() - started

    after = bookings if negotiation.agreement is None else (*bookings, negotiation.agreement.offer.booking)
    return dict(spec=spec, admission=admission, offers=offers, negotiation=negotiation, result=result,
                bookings_before=bookings, bookings_after=after, seconds=seconds,
                outcome=classify(request, admission, offers, negotiation),
                **check_attempt(spec, admission, offers, negotiation, result, commitment, ledger, bookings, after))


def run_day(specs, initial=INITIAL):
    commitment, bookings, ledger = standing(*initial), tuple(initial), ()
    records = []
    for spec in specs:
        record = attempt(spec, commitment, bookings, ledger)
        records.append(record)
        commitment, bookings, ledger = record['result'].commitment, record['bookings_after'], record['result'].ledger
    return records

## Deliberate boundary cases

Random sampling rarely hits these exactly. Each case runs once from the initial commitments
(A 08:00–09:00, B 10:30–11:30) with the same checks. Unless stated, a case is received at 07:50 with $(c,v)=(2,8)$.
The expected outcome is asserted **after** running the method.

In [5]:
def case(label, service, alternatives=(), client=9, received=470, cost=2, value=8, *, outcome, starts, offers):
    spec = dict(received=F(received), request=Request(client, service, alternatives), cost=F(cost),
                value=F(value), targets=TARGETS, minima=MINIMA)
    return dict(label=label, spec=spec, outcome=OUTCOMES[outcome], starts=starts, offers=tuple(map(F, offers)))

BOUNDARY = (
    case('Adjacent after A', Service(P.singleton(540), 30), outcome=0, starts=P.singleton(540), offers=[540]),
    case('Adjacent before B', Service(P.singleton(600), 30), outcome=0, starts=P.singleton(600), offers=[600]),
    case('Exact fit between A and B', Service(P.closed(540, 600), 90), outcome=0, starts=P.singleton(540), offers=[540]),
    case('One minute too long', Service(P.closed(540, 600), 91), outcome=2, starts=P.empty(), offers=[]),
    case('Fully blocked, alternative too', Service(P.closed(630, 660), 60), (Service(P.closed(630, 660), 30),),
         outcome=2, starts=P.empty(), offers=[]),
    case('Blocked; alternative fits exactly', Service(P.closed(600, 660), 60), (Service(P.closed(600, 660), 30),),
         outcome=1, starts=P.singleton(600), offers=[600]),
    case('Same-client reservation (A)', Service(P.closed(500, 520), 30), client=1,
         outcome=0, starts=P.closed(500, 520), offers=[500, 515]),
    case('Same window, other client', Service(P.closed(500, 520), 30), outcome=2, starts=P.empty(), offers=[]),
    case('Narrow open window', Service(P.open(540, 560), 60), outcome=3, starts=P.open(540, 560), offers=[]),
    case('Open window, closed feasible end', Service(P.open(540, 600), 60),
         outcome=0, starts=P.openclosed(540, 570), offers=[570]),
    case('Window partly in the past', Service(P.closed(590, 620), 30), received=600,
         outcome=0, starts=P.singleton(600), offers=[600]),
    case('Disjoint price ranges', Service(P.closed(720, 750), 60), cost=4, value=4,
         outcome=4, starts=P.closed(720, 750), offers=[720, 750]),
    case('Overlap, budget too short', Service(P.closed(720, 750), 60), cost=2, value=5,
         outcome=4, starts=P.closed(720, 750), offers=[720, 750]),
    case('Agreement in the final round', Service(P.closed(720, 750), 60), cost=2, value=6,
         outcome=0, starts=P.closed(720, 750), offers=[720, 750]),
)

boundary = [run_day((c['spec'],))[0] for c in BOUNDARY]
for c, r in zip(BOUNDARY, boundary):
    starts = r['admission'].feasible_starts if r['admission'] else P.empty()
    assert (r['outcome'], starts, tuple(o.start for o in r['offers'])) == (c['outcome'], c['starts'], c['offers']), c['label']
    assert not r['disagreement'] and not any(r['violations'].values()), c['label']
assert len(boundary[-1]['negotiation'].history) == ROUND_BUDGET and boundary[-1]['negotiation'].agreement

def service_text(service):
    return f"{int(service.duration)} min from {clock_interval(service.starts)}"

table(['Case', 'Requested service', 'Alternative', 'Feasible starts', 'Offer starts', 'Outcome', 'Rounds', 'Check'],
      [(c['label'], service_text(r['spec']['request'].service),
        ', '.join(service_text(s) for s in r['spec']['request'].alternatives) or '—',
        clock_interval(r['admission'].feasible_starts) if r['admission'] else '∅',
        ', '.join(clock(o.start) for o in r['offers']) or '—', r['outcome'],
        len(r['negotiation'].history), 'PASS') for c, r in zip(BOUNDARY, boundary)])

| Case | Requested service | Alternative | Feasible starts | Offer starts | Outcome | Rounds | Check |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Adjacent after A | 30 min from {09:00} | — | {09:00} | 09:00 | Agreement · full service | 2 | PASS |
| Adjacent before B | 30 min from {10:00} | — | {10:00} | 10:00 | Agreement · full service | 2 | PASS |
| Exact fit between A and B | 90 min from [09:00,10:00] | — | {09:00} | 09:00 | Agreement · full service | 2 | PASS |
| One minute too long | 91 min from [09:00,10:00] | — | ∅ | — | Failure · infeasible service | 0 | PASS |
| Fully blocked, alternative too | 60 min from [10:30,11:00] | 30 min from [10:30,11:00] | ∅ | — | Failure · infeasible service | 0 | PASS |
| Blocked; alternative fits exactly | 60 min from [10:00,11:00] | 30 min from [10:00,11:00] | {10:00} | 10:00 | Agreement · adapted service | 2 | PASS |
| Same-client reservation (A) | 30 min from [08:20,08:40] | — | [08:20,08:40] | 08:20, 08:35 | Agreement · full service | 2 | PASS |
| Same window, other client | 30 min from [08:20,08:40] | — | ∅ | — | Failure · infeasible service | 0 | PASS |
| Narrow open window | 60 min from (09:00,09:20) | — | (09:00,09:20) | — | Failure · empty sampled offer set | 0 | PASS |
| Open window, closed feasible end | 60 min from (09:00,10:00) | — | (09:00,09:30] | 09:30 | Agreement · full service | 2 | PASS |
| Window partly in the past | 30 min from [09:50,10:20] | — | {10:00} | 10:00 | Agreement · full service | 2 | PASS |
| Disjoint price ranges | 60 min from [12:00,12:30] | — | [12:00,12:30] | 12:00, 12:30 | Failure · exhausted bargaining | 3 | PASS |
| Overlap, budget too short | 60 min from [12:00,12:30] | — | [12:00,12:30] | 12:00, 12:30 | Failure · exhausted bargaining | 3 | PASS |
| Agreement in the final round | 60 min from [12:00,12:30] | — | [12:00,12:30] | 12:00, 12:30 | Agreement · full service | 3 | PASS |

## Synthetic days

20 seeds × three load levels: 1,400 sequential attempts. Each day starts again from the initial commitments.

In [6]:
days = {load: [run_day(synthetic_day(load, seed)) for seed in SEEDS] for load in LOADS}
records = {load: [r for day in rs for r in day] for load, rs in days.items()}

## Results

- **Figure:** outcome shares per load level. The three failure kinds stay separate:
  *infeasible service* (no start for any permitted service), *empty sampled offer set*
  (feasible starts, but none on the sampling grid) and *exhausted bargaining* (certified offers, no agreement within three rounds).
- **Table:** checks per attempt and timing of the four method calls; boundary cases form the last row.
  *Invariant violations* counts attempts with at least one violated invariant.
- **Reading:** infeasible service grows with load as accepted bookings fill the horizon. Adapted agreements
  are rare: halving $d$ helps only when a request is blocked at $d$ but not at $d/2$ within the same window.
  Exhausted bargaining depends on the declared $(c,v)$ draw, not on load. Execution time grows with
  the number of standing commitments.

In [7]:
figure = plot_outcome_shares({load: Counter(r['outcome'] for r in rs) for load, rs in records.items()}, OUTCOMES)
display_timeline(figure)

Attempt outcomes by load level

In [8]:
def summary(label, rs):
    ms = [1000 * r['seconds'] for r in rs]
    return (label, len(rs), sum(r['disagreement'] for r in rs),
            sum(any(r['violations'].values()) for r in rs),
            f'{median(ms):.0f} ms', f"{quantiles(ms, n=20, method='inclusive')[-1]:.0f} ms")

table(['Workload', 'Requests', 'Feasibility-check disagreements', 'Invariant violations', 'Median time', '95th-percentile time'],
      [*(summary(f'{load} requests/day × {len(SEEDS)} seeds', rs) for load, rs in records.items()),
       summary('Boundary cases', boundary)])

| Workload | Requests | Feasibility-check disagreements | Invariant violations | Median time | 95th-percentile time |
| --- | --- | --- | --- | --- | --- |
| 10 requests/day × 20 seeds | 200 | 0 | 0 | 57 ms | 102 ms |
| 20 requests/day × 20 seeds | 400 | 0 | 0 | 66 ms | 116 ms |
| 40 requests/day × 20 seeds | 800 | 0 | 0 | 77 ms | 152 ms |
| Boundary cases | 14 | 0 | 0 | 38 ms | 63 ms |

## Result checks

These assert, after the runs, that every check passed and that the workload covered what it declares.
They do not select outcomes.

- With closed windows, every feasible component is left-closed, so its first grid point is an offer.
  An empty sampled offer set therefore occurs only for open windows, in the boundary cases.

In [9]:
attempts = [r for rs in records.values() for r in rs] + boundary
violations = Counter(name for r in attempts for name, bad in r['violations'].items() if bad)
assert not violations and not any(r['disagreement'] for r in attempts), violations

def overlapping(spec):
    return spec['cost'] + spec['minima']['owner'] <= spec['value'] - spec['minima']['client']

for load, rs in records.items():
    assert {overlapping(r['spec']) for r in rs} == {True, False}, load
    assert Counter(r['outcome'] for r in rs)[OUTCOMES[3]] == 0
    assert not any(r['negotiation'].agreement and not overlapping(r['spec']) for r in rs)
    assert all(len(day[-1]['bookings_after']) == len(INITIAL) + sum(bool(r['negotiation'].agreement) for r in day)
               for day in days[load])

# Both kinds of price range reach bargaining, and both can exhaust it.
exhausted = Counter(overlapping(r['spec']) for rs in records.values() for r in rs if r['outcome'] == OUTCOMES[4])
assert exhausted[True] and exhausted[False]
print(f"{len(attempts)} attempts × {len(attempts[0]['violations'])} invariants: 0 violations, 0 feasibility disagreements.")
print(f'Exhausted bargaining: {exhausted[False]} with disjoint price ranges, '
      f'{exhausted[True]} with overlapping ranges but too few rounds.')

1414 attempts × 6 invariants: 0 violations, 0 feasibility disagreements.
Exhausted bargaining: 47 with disjoint price ranges, 98 with overlapping ranges but too few rounds.
